# SVM Poisoning Defenses
This notebook evaluates Brute Force RONI, Fast RONI (Influence Functions), and Data Sanitization defenses against SVM poisoning attacks on 2D Gaussian and MNIST datasets.
We inject `NUMBER_OF_POISON_INSTANCES = 10` poison points in each experiment and print validation/test loss and error.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
import time
import warnings
warnings.filterwarnings('ignore')

# Import functions from the existing svm_poisoning.py
from svm_poisoning import (
    LinearKernel, RBFKernel, PolynomialKernel, train_svm, poisoning_attack, 
    get_svm_info, load_mnist, get_binary_problem, SEED
)
np.random.seed(SEED)

NUMBER_OF_POISON_INSTANCES = 10


In [ ]:
def print_stats(name, clf, X_val, y_val, X_test=None, y_test=None):
    print(f"--- Stats for {name} ---")
    val_loss = np.mean(np.maximum(0, 1 - y_val * clf.decision_function(X_val)))
    val_err = np.mean(clf.predict(X_val) != y_val)
    print(f"Validation Hinge Loss : {val_loss:.4f}")
    print(f"Validation Error      : {val_err:.4f}")
    
    if X_test is not None:
        test_loss = np.mean(np.maximum(0, 1 - y_test * clf.decision_function(X_test)))
        test_err = np.mean(clf.predict(X_test) != y_test)
        print(f"Test Hinge Loss       : {test_loss:.4f}")
        print(f"Test Error            : {test_err:.4f}")
    print()


In [ ]:
def brute_force_roni_defense(X_train, y_train, X_val, y_val, kernel, C=1.0, threshold_k=1.5):
    """
    Implements RONI (Reject On Negative Impact) using brute-force leave-one-out.
    """
    n = len(X_train)
    clf_clean = train_svm(X_train, y_train, C, kernel)
    L_clean = np.mean(np.maximum(0, 1 - y_val * clf_clean.decision_function(X_val)))
    
    impacts = np.zeros(n)
    for i in range(n):
        X_minus_i = np.delete(X_train, i, axis=0)
        y_minus_i = np.delete(y_train, i)
        clf_i = train_svm(X_minus_i, y_minus_i, C, kernel)
        L_without_i = np.mean(np.maximum(0, 1 - y_val * clf_i.decision_function(X_val)))
        impacts[i] = L_clean - L_without_i
        
    mean_impact = np.mean(impacts)
    std_impact = np.std(impacts)
    threshold = mean_impact + threshold_k * std_impact
    suspicious_indices = np.where(impacts > threshold)[0]
    
    X_sanitized = np.delete(X_train, suspicious_indices, axis=0)
    y_sanitized = np.delete(y_train, suspicious_indices)
    
    return X_sanitized, y_sanitized, suspicious_indices, impacts

def fast_roni_defense(X_train, y_train, X_val, y_val, kernel, C=1.0, threshold_k=1.5):
    """
    Fast RONI using Influence Functions (first-order approximation of leave-one-out).
    """
    n = len(X_train)
    clf = train_svm(X_train, y_train, C, kernel)
    info = get_svm_info(clf, X_train, y_train, C, attack_idx=0, kernel=kernel)
    
    alpha = info['all_alphas']
    b = info['b']
    S_indices = info['S_indices']
    X_s = info['X_s']
    y_s = info['y_s']
    n_s = len(S_indices)
    
    f_val = clf.decision_function(X_val)
    L_clean = np.mean(np.maximum(0, 1 - y_val * f_val))
    
    impacts = np.zeros(n)
    
    if n_s == 0:
        return X_train, y_train, [], impacts
        
    K_ss = kernel.compute(X_s, X_s)
    Q_ss = np.outer(y_s, y_s) * K_ss
    Q_ss += 1e-9 * np.eye(n_s)
    
    try:
        Q_ss_inv = np.linalg.inv(Q_ss)
    except np.linalg.LinAlgError:
        return X_train, y_train, [], impacts
        
    v = Q_ss_inv @ y_s
    zeta = float(y_s @ v)
    if abs(zeta) < 1e-12:
        return X_train, y_train, [], impacts
        
    K_val_s = kernel.compute(X_val, X_s)
    
    for i in range(n):
        if alpha[i] < 1e-8:
            impacts[i] = 0.0
            continue
            
        d_alpha_i = -alpha[i]
        
        x_i = X_train[i:i+1]
        y_i = y_train[i]
        
        K_si = kernel.compute(X_s, x_i).flatten()
        Q_si = y_s * y_i * K_si
        
        db = - (1.0 / zeta) * (np.dot(v, Q_si) - y_i) * d_alpha_i
        d_alpha_S = - Q_ss_inv @ (Q_si * d_alpha_i + y_s * db)
        
        term1 = K_val_s @ (d_alpha_S * y_s)
        term2 = db
        K_val_i = kernel.compute(X_val, x_i).flatten()
        term3 = d_alpha_i * y_i * K_val_i
        
        df_val = term1 + term2 + term3
        f_val_new = f_val + df_val
        
        L_new = np.mean(np.maximum(0, 1 - y_val * f_val_new))
        impacts[i] = L_clean - L_new

    mean_impact = np.mean(impacts)
    std_impact = np.std(impacts)
    threshold = mean_impact + threshold_k * std_impact
    suspicious_indices = np.where(impacts > threshold)[0]
    
    X_sanitized = np.delete(X_train, suspicious_indices, axis=0)
    y_sanitized = np.delete(y_train, suspicious_indices)
    
    return X_sanitized, y_sanitized, suspicious_indices, impacts

def data_sanitization_defense(X_train, y_train, kernel, k=1.5):
    """
    Detects outliers in feature/kernel space by measuring distance to class centroid.
    """
    n = len(X_train)
    suspicious_indices = []
    distances = np.zeros(n)
    
    for c in [-1, 1]:
        class_indices = np.where(y_train == c)[0]
        if len(class_indices) == 0: continue
            
        X_c = X_train[class_indices]
        K_cc = kernel.compute(X_c, X_c)
        term3 = np.sum(K_cc) / (len(X_c) ** 2)
        
        d_c = np.zeros(len(class_indices))
        for idx, i in enumerate(class_indices):
            x_i = X_train[i:i+1]
            term1 = kernel.compute(x_i, x_i)[0, 0]
            term2 = 2.0 * np.sum(kernel.compute(x_i, X_c)) / len(X_c)
            d_c[idx] = term1 - term2 + term3
            distances[i] = d_c[idx]
            
        mean_d = np.mean(d_c)
        std_d = np.std(d_c)
        threshold = mean_d + k * std_d
        
        outliers = class_indices[d_c > threshold]
        suspicious_indices.extend(outliers)
        
    suspicious_indices = np.array(suspicious_indices, dtype=int)
    X_sanitized = np.delete(X_train, suspicious_indices, axis=0)
    y_sanitized = np.delete(y_train, suspicious_indices)
    
    return X_sanitized, y_sanitized, suspicious_indices, distances


In [ ]:
def run_exp1_defenses():
    print("="*50)
    print("EXPERIMENT 1: 2D Gaussian")
    print("="*50)
    n_tr = 25
    n_val = 500
    cov = 0.6 * np.eye(2)
    X_neg_tr = np.random.multivariate_normal([-1.5, 0], cov, n_tr)
    X_pos_tr = np.random.multivariate_normal([1.5, 0], cov, n_tr)
    X_tr = np.vstack([X_neg_tr, X_pos_tr])
    y_tr = np.concatenate([-np.ones(n_tr), np.ones(n_tr)])
    
    X_neg_val = np.random.multivariate_normal([-1.5, 0], cov, n_val)
    X_pos_val = np.random.multivariate_normal([1.5, 0], cov, n_val)
    X_val = np.vstack([X_neg_val, X_pos_val])
    y_val = np.concatenate([-np.ones(n_val), np.ones(n_val)])
    
    kernel = LinearKernel()
    C = 1.0
    
    clf_clean = train_svm(X_tr, y_tr, C, kernel)
    print_stats("Clean Model", clf_clean, X_val, y_val)
    
    print(f"Injecting {NUMBER_OF_POISON_INSTANCES} poison points...")
    X_tr_poisoned = X_tr.copy()
    y_tr_poisoned = y_tr.copy()
    yc = -1
    pos_indices = np.where(y_tr == 1)[0]
    
    for _ in range(NUMBER_OF_POISON_INSTANCES):
        init_idx = np.random.choice(pos_indices)
        xc_init = X_tr[init_idx].copy()
        res = poisoning_attack(
            X_tr_poisoned, y_tr_poisoned, X_val, y_val, yc, xc_init, kernel,
            C=C, step_size=0.05, max_iter=100, epsilon=1e-4, patience=5,
            gradient_sign=-1, verbose=False, print_every=0,
            bound_min=np.array([-4.0, -4.0]), bound_max=np.array([4.0, 4.0])
        )
        X_tr_poisoned = np.vstack([X_tr_poisoned, res['xc'].reshape(1, -1)])
        y_tr_poisoned = np.concatenate([y_tr_poisoned, [yc]])
        
    clf_poisoned = train_svm(X_tr_poisoned, y_tr_poisoned, C, kernel)
    print_stats("Poisoned Model", clf_poisoned, X_val, y_val)
    
    print("Running Fast RONI (Influence Functions) defense...")
    X_roni, y_roni, r_idx, impacts = fast_roni_defense(X_tr_poisoned, y_tr_poisoned, X_val, y_val, kernel, C, threshold_k=1.5)
    clf_roni = train_svm(X_roni, y_roni, C, kernel)
    print(f"Points removed by Fast RONI: {len(r_idx)} (Poison points are at indices >= {len(X_tr)})")
    print_stats("Fast RONI Defended Model", clf_roni, X_val, y_val)
    
    print("Running Data Sanitization defense...")
    X_san, y_san, s_idx, dists = data_sanitization_defense(X_tr_poisoned, y_tr_poisoned, kernel, k=1.5)
    clf_san = train_svm(X_san, y_san, C, kernel)
    print(f"Points removed by Data Sanitization: {len(s_idx)} (Poison points are at indices >= {len(X_tr)})")
    print_stats("Data Sanitization Defended Model", clf_san, X_val, y_val)

run_exp1_defenses()


In [ ]:
def run_exp2_defenses():
    print("="*50)
    print("EXPERIMENT 2: MNIST (7 vs 1)")
    print("="*50)
    
    X_mnist, y_mnist = load_mnist()
    X_tr, y_tr, X_val, y_val, X_test, y_test = get_binary_problem(X_mnist, y_mnist, 7, 1)
    
    kernel = LinearKernel()
    C = 1.0
    
    clf_clean = train_svm(X_tr, y_tr, C, kernel)
    print_stats("Clean Model", clf_clean, X_val, y_val, X_test, y_test)
    
    print(f"Injecting {NUMBER_OF_POISON_INSTANCES} poison points...")
    X_tr_poisoned = X_tr.copy()
    y_tr_poisoned = y_tr.copy()
    yc = -1
    pos_indices = np.where(y_tr == 1)[0]
    
    for i in range(NUMBER_OF_POISON_INSTANCES):
        print(f"  Optimizing poison point {i+1}/{NUMBER_OF_POISON_INSTANCES}...")
        init_idx = np.random.choice(pos_indices)
        xc_init = X_tr[init_idx].copy()
        res = poisoning_attack(
            X_tr_poisoned, y_tr_poisoned, X_val, y_val, yc, xc_init, kernel,
            C=C, step_size=0.5, max_iter=100, epsilon=0.01,
            gradient_sign=-1, verbose=False, print_every=0,
            bound_min=np.zeros(784), bound_max=np.ones(784)
        )
        X_tr_poisoned = np.vstack([X_tr_poisoned, res['xc'].reshape(1, -1)])
        y_tr_poisoned = np.concatenate([y_tr_poisoned, [yc]])
        
    clf_poisoned = train_svm(X_tr_poisoned, y_tr_poisoned, C, kernel)
    print_stats("Poisoned Model", clf_poisoned, X_val, y_val, X_test, y_test)
    
    print("Running Fast RONI (Influence Functions) defense (this should be quick!)...")
    X_roni, y_roni, r_idx, impacts = fast_roni_defense(X_tr_poisoned, y_tr_poisoned, X_val, y_val, kernel, C, threshold_k=1.5)
    clf_roni = train_svm(X_roni, y_roni, C, kernel)
    print(f"Points removed by Fast RONI: {len(r_idx)} (Poison points are at indices >= {len(X_tr)})")
    print_stats("Fast RONI Defended Model", clf_roni, X_val, y_val, X_test, y_test)
    
    print("Running Data Sanitization defense...")
    X_san, y_san, s_idx, dists = data_sanitization_defense(X_tr_poisoned, y_tr_poisoned, kernel, k=1.5)
    clf_san = train_svm(X_san, y_san, C, kernel)
    print(f"Points removed by Data Sanitization: {len(s_idx)} (Poison points are at indices >= {len(X_tr)})")
    print_stats("Data Sanitization Defended Model", clf_san, X_val, y_val, X_test, y_test)

run_exp2_defenses()
